# Escalado del semillerío 9105 a k=100

Agrega 50 semillas nuevas al semillerío del 9105 (reusa las 50 que ya están en `./semillas/`). Es la mejora más limpia que queda: matemáticamente sd/√n → sd de ~0.6 (k=50) baja a ~0.4 (k=100).

**Requisito:** el pipeline del 9105 debe estar cargado en memoria (`dfinal_train`, `param_final` con num_leaves=256, `campos_buenos`, `mfuture`, `dfuture`). Si perdiste esa sesión, correr primero el 9105 hasta que arme `param_final`.

In [ ]:
# --- Sanity check: objetos del 9105 en memoria ---
require("data.table")
require("lightgbm")

obligatorios <- c("dfinal_train", "param_final", "campos_buenos", "mfuture", "dfuture")
faltantes <- obligatorios[!sapply(obligatorios, exists)]

if (length(faltantes) > 0) {
  stop("Faltan objetos en memoria: ", paste(faltantes, collapse = ", "),
       "\nCorré el notebook 9105 hasta la celda que arma param_final antes de continuar.")
}

cat("Todos los objetos necesarios están en memoria.\n")
cat("num_iterations final: ", param_final$num_iterations, "\n")
cat("num_leaves final:     ", param_final$num_leaves, "\n")
cat("learning_rate final:  ", param_final$learning_rate, "\n")
cat("(esperados 9105: niter=347, leaves=256, lr=0.01318)\n")

In [ ]:
# --- Definición de las 100 semillas: 50 ya en disco + 50 nuevas ---

PARAM$semillerio$semillas_k50 <- c(
  # 20 originales
  804043, 653561, 703903, 439693, 665857,
  246319, 719179, 688511, 678859, 759179,
  748567, 319687, 771091, 684007, 514853,
  377749, 329977, 757927, 724837, 216973,
  # 30 del salto k=20 → k=50
  287333, 656119, 694919, 189817, 867463,
  791801, 804317, 680831, 330917, 595951,
  777571, 662339, 202667, 526159, 509389,
  865993, 749471, 398833, 153269, 969637,
  374683, 678481, 799333, 687083, 941207,
  213349, 735659, 872843, 803207, 414913
)

# 50 semillas NUEVAS del banco de primos (siguientes 50 sin usar)
PARAM$semillerio$semillas_nuevas50 <- c(
  400853, 708161, 563249, 425501, 598219,
  502633, 821027, 105817, 643121, 940349,
  954599, 345887, 535973, 165293, 486817,
  271723, 810253, 934907, 869443, 782057,
  472063, 110647, 682147, 384079, 430879,
  661897, 157177, 768857, 705053, 819799,
  418597, 336653, 564229, 755903, 264139,
  592741, 576089, 422557, 804941, 765623,
  195677, 402221, 519907, 101209, 752881,
  755791, 402023, 355573, 396581, 568241
)

PARAM$semillerio$semillas_k100 <- c(
  PARAM$semillerio$semillas_k50,
  PARAM$semillerio$semillas_nuevas50
)

stopifnot(length(PARAM$semillerio$semillas_k100) == 100)
stopifnot(length(unique(PARAM$semillerio$semillas_k100)) == 100)

dir.create("semillas", showWarnings = FALSE)
ya_entrenadas <- sapply(PARAM$semillerio$semillas_k100, function(s) {
  file.exists(paste0("semillas/prediccion_semilla_", s, ".txt"))
})

cat("Ya en disco: ", sum(ya_entrenadas), "\n")
cat("A entrenar:  ", sum(!ya_entrenadas), "\n")
cat("(esperado: 50 en disco, 50 a entrenar)\n")

In [ ]:
# --- Loop de entrenamiento de las 50 semillas nuevas ---
# Estimado: 50 semillas × ~1-1.5 min c/u = ~50-75 min

semillas_pendientes <- PARAM$semillerio$semillas_k100[!ya_entrenadas]

for (i in seq_along(semillas_pendientes)) {

  semilla <- semillas_pendientes[i]
  cat(format(Sys.time(), "%X"),
      " - Nueva semilla ", i, "/", length(semillas_pendientes),
      " = ", semilla, "\n", sep = "")

  param_semilla <- param_final
  param_semilla$seed <- semilla

  modelo_i <- lgb.train(
    data = dfinal_train,
    param = param_semilla,
    verbose = -100
  )

  prob_i <- predict(modelo_i, mfuture)

  tb_pred_i <- dfuture[, list(numero_de_cliente)]
  tb_pred_i[, prob := prob_i]
  fwrite(tb_pred_i,
    file = paste0("semillas/prediccion_semilla_", semilla, ".txt"),
    sep = "\t"
  )

  rm(modelo_i, prob_i, tb_pred_i)
  gc(full = TRUE, verbose = FALSE)
}

cat("\nEntrenamiento completado. Las 100 semillas del 9105 están en disco.\n")

In [ ]:
# --- Cargar las 100 predicciones y armar ensemble k=100 ---

tb_probs100 <- dfuture[, list(numero_de_cliente)]

for (semilla in PARAM$semillerio$semillas_k100) {
  tb_ind <- fread(paste0("semillas/prediccion_semilla_", semilla, ".txt"))
  tb_ind <- tb_ind[match(tb_probs100$numero_de_cliente, tb_ind$numero_de_cliente)]
  col_semilla <- paste0("prob_", semilla)
  tb_probs100[, (col_semilla) := tb_ind$prob]
}

cols_prob100 <- grep("^prob_", colnames(tb_probs100), value = TRUE)
stopifnot(length(cols_prob100) == 100)

tb_prediccion_k100 <- tb_probs100[, list(numero_de_cliente)]
tb_prediccion_k100[, prob := rowMeans(tb_probs100[, ..cols_prob100])]

fwrite(tb_prediccion_k100, file = "prediccion_k100.txt", sep = "\t")

# Comparación con k=50 (solo las 50 originales)
cols_prob50 <- paste0("prob_", PARAM$semillerio$semillas_k50)
tb_pred50 <- tb_probs100[, list(numero_de_cliente)]
tb_pred50[, prob := rowMeans(tb_probs100[, ..cols_prob50])]

cat("=== Análisis k=50 vs k=100 ===\n")
cor_50_100 <- cor(tb_pred50$prob, tb_prediccion_k100$prob)
cat("Correlación entre ensembles k=50 y k=100:", round(cor_50_100, 5), "\n")

corte_ref <- 2000
top_k100 <- {tmp <- copy(tb_prediccion_k100); setorder(tmp, -prob); tmp[1:corte_ref, numero_de_cliente]}
top_k50 <- {tmp <- copy(tb_pred50); setorder(tmp, -prob); tmp[1:corte_ref, numero_de_cliente]}

en_ambos <- length(intersect(top_k100, top_k50))
cat("Coincidencia top-", corte_ref, " entre k=50 y k=100:",
    en_ambos, "/", corte_ref, "(", round(en_ambos/corte_ref, 3), ")\n")

In [ ]:
# --- Submit del ensemble k=100 a Kaggle ---
# Solo 5 cortes centrales (1800-2200) para verificar meseta

PARAM$kaggle$competencia <- "data-mining-junior-2026-a"
PARAM$kaggle$cortes <- seq(1800, 2200, by = 100)

setorder(tb_prediccion_k100, -prob)
dir.create("kaggle", showWarnings = FALSE)

for (envios in PARAM$kaggle$cortes) {

  tb_prediccion_k100[, Predicted := 0L]
  tb_prediccion_k100[1:envios, Predicted := 1L]

  archivo_kaggle <- paste0("./kaggle/KA9105_FEratios_k100_", envios, ".csv")

  fwrite(tb_prediccion_k100[, list(numero_de_cliente, Predicted)],
         file = archivo_kaggle, sep = ",")

  comando <- "kaggle competitions submit"
  competencia <- paste("-c", PARAM$kaggle$competencia)
  arch <- paste("-f", archivo_kaggle)
  mensaje <- paste0("-m '9105 FE+ratios k=100 envios=", envios, "'")

  linea <- paste(comando, competencia, arch, mensaje)

  cat(format(Sys.time(), "%X"), " - submit k=100 envios=", envios, "\n", sep = "")
  salida <- system(linea, intern = TRUE)
  Sys.sleep(30)
  cat(salida, "\n")
}

cat("\nSubmits del ensemble k=100 completados.\n")